<h1>🎛️ Biofilter — Report: <code>pair_variants</code></h1>

Candidate variant × variant pairs whose genes share biology.

**It takes variants** — rsIDs, `chr:pos`, `chr:pos:ref:alt` — and pairs
the ones you named. Section 5 is about what it deliberately no longer
does, and how to get it back in one visible step.

### 1. Open a bundle, and get some variants

In [ ]:
from pathlib import Path

from biofilter import Biofilter
from biofilter.modules.report import Bundle

BUNDLE = None
REPORT = "pair_variants"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# A realistic input: common variants in three genes that share biology.
with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    VARIANTS = [r[0] for r in bundle.con.execute("""
        SELECT v.variant_key
        FROM variant_masters v
        JOIN entity_locations l ON l.build = 38 AND l.chromosome = v.chromosome
         AND v.position BETWEEN l.start_pos AND l.end_pos
        JOIN gene_masters gm ON gm.entity_id = l.entity_id
        WHERE gm.symbol IN ('CHEK2', 'SMARCB1', 'NF2') AND v.af_joint > 0.05
        ORDER BY v.af_joint DESC LIMIT 40
    """).fetchall()]

print(f"{len(VARIANTS)} variants, e.g. {VARIANTS[:2]}")

### 2. What can link two genes in this bundle

A *group* is the entity sitting between two genes: a pathway they share,
a disease both are implicated in, a protein both interact with.

In [ ]:
with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    groups = bundle.con.execute("""
        WITH ge AS (SELECT e.id FROM entities e JOIN entity_groups eg ON eg.id = e.group_id
                    WHERE eg.name = 'Genes'),
        link AS (
            SELECT r.entity_1_id AS grp, r.entity_2_id AS gene FROM entity_relationships r
            WHERE r.entity_2_id IN (SELECT id FROM ge) AND r.entity_1_id NOT IN (SELECT id FROM ge)
            UNION ALL
            SELECT r.entity_2_id, r.entity_1_id FROM entity_relationships r
            WHERE r.entity_1_id IN (SELECT id FROM ge) AND r.entity_2_id NOT IN (SELECT id FROM ge))
        SELECT eg.name AS group_type, count(DISTINCT l.grp) AS groups
        FROM link l JOIN entities e ON e.id = l.grp
        JOIN entity_groups eg ON eg.id = e.group_id
        GROUP BY 1 ORDER BY 2 DESC
    """).to_arrow_table().to_pandas()

groups

### 3. Pair them

One row is a pair of your variants whose genes share at least one
group.

In [ ]:
pairs = bf.report.run(REPORT, input_data=VARIANTS, group_types=["Proteins"])
df = pairs.to_pandas()

print(f"{len(df):,} pairs from {len(VARIANTS)} variants")
df[["variant_1_key", "gene_1_symbol", "variant_2_key", "gene_2_symbol",
    "group_support_count", "group_support_sources"]].head(6)

### 4. `max_group_size`, and why an empty result is not "no biology"

A pathway naming 2,615 genes links its members while saying almost
nothing about any of them. When a result comes back empty or thin, this
is usually why — so the provenance says what the cut removed.

In [ ]:
for size in (200, 300, 1000, 0):
    out = bf.report.run(REPORT, input_data=VARIANTS, group_types=["Proteins"],
                        max_group_size=size)
    label = "no limit" if size == 0 else str(size)
    print(f"  max_group_size={label:<9} {out.num_rows:>7,} pairs")

pairs.provenance["group_filter"]

### 5. Starting from genes

This report used to accept gene names and expand each one into the
variants inside it. It no longer does, and the reason is not tidiness.

A gene on chr22 holds about 4,000 variants. The report kept **100** of
them, ranked by allele frequency, and you never saw which 100. Running
the expansion yourself costs one step and puts that choice in front of
the person making it.

In [ ]:
try:
    bf.report.run(REPORT, input_data=["CHEK2", "SMARCB1"])
except ValueError as exc:
    print(exc)

In [ ]:
# One visible step instead of one invisible one — and you can look at
# and filter the list before anything is paired.
expanded = bf.report.run("expand_gene_to_variant",
                         input_data=["CHEK2", "SMARCB1"],
                         mapping="position", af_max=0.01,
                         impact_filter=["HIGH", "MODERATE"],
                         max_variants_per_gene=0).to_pandas()

keys = expanded.query("status == 'ok'").variant_key.drop_duplicates().tolist()
print(f"{len(keys):,} rare, damaging variants — yours to filter further")

from_genes = bf.report.run(REPORT, input_data=keys[:200], group_types=["Proteins"])
print(f"{from_genes.num_rows:,} pairs")

Three parameters left with it. Passing one is an error rather than a
silent change of answer:

| gone | instead |
| --- | --- |
| gene names in `input_data` | `expand_gene_to_variant`, then pair its output |
| `max_variants_per_gene` | the same parameter on `expand_gene_to_variant` |
| `membership="either"` | `pair_genes(membership="either")` → expand the partners → pair |

In [ ]:
for name, value in [("membership", "either"),
                    ("max_variants_per_gene", 10),
                    ("output_grain", "gene_pairs")]:
    try:
        bf.report.run(REPORT, input_data=VARIANTS[:4], **{name: value})
    except ValueError as exc:
        print(f"{name}: {str(exc).splitlines()[-1].strip()}\n")

### 6. Support, and what it is not

`group_support_count` counts the groups linking the two genes;
`group_support_source_count` counts the **curations** that asserted them,
read from the bundle rather than guessed from an accession prefix. Two
curations agreeing is not one curation saying it twice.

In [ ]:
for support in (1, 10, 30):
    out = bf.report.run(REPORT, input_data=VARIANTS, group_types=["Proteins"],
                        min_group_support=support).to_pandas()
    print(f"  min_group_support={support:>3}  {len(out):>7,} pairs")

df.nlargest(5, "group_support_count")[
    ["gene_1_symbol", "gene_2_symbol", "group_support_count",
     "group_support_source_count", "group_support_sources"]
]

It is a weight for ranking candidates — not a p-value, and not evidence
of interaction. Change `max_group_size` and every count changes.

### 7. Gene pairs are a different question

Stage 2 on its own — which genes are related, and by what — is
`pair_genes`. It also expands a gene pair by a list **you** supply, which
is what to reach for when the variant-to-gene attachment comes from
outside the bundle: a colocalization, a fine-mapping, a curated
assignment.

This report derives that attachment from coordinates, which is right for
a coding variant and wrong for a regulatory one.

In [ ]:
partners = bf.report.run("pair_genes", input_data=["CHEK2"],
                         group_types=["Proteins"], membership="either").to_pandas()

print(f"{len(partners):,} partner genes for CHEK2")
partners[["gene_1_symbol", "gene_2_symbol", "gene_2_from_input",
          "group_support_count", "group_support_sources"]].head(6)

### 8. Export

In [ ]:
for path in pairs.write(OUTPUT_DIR / "pair_variants.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter report run --report-name pair_variants \\
    --input-file my_variants.txt \\
    --param group_types=Proteins \\
    --param max_group_size=300 \\
    --param min_group_sources=2 \\
    --output pairs.csv
```